# 14 · Ensembles — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch14_ensembles/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch14_ensembles/03_notebook.ipynb`, créée par `python tools/start_chapter.py 14`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 14.Q1 à 14.Q11, 14.R1 à 14.R3, 14.1 à 14.3, 14.5 à 14.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 14.11 à 14.14 | faire voter trois classifieurs du ch. 13, le vote et le bootstrap dans mylearn.ensemble, un jury de votants indépendants puis corrélés, voting et stacking (*prochaine session de génération*) | | | |
| B | 14.15 à 14.19 | forêts de scikit-learn sur California, nombre d'arbres face au bruit, BaggingClassifier, RandomForestClassifier et ExtraTreesClassifier dans mylearn.ensemble, importance des features (*prochaine session de génération*) | | | |
| C | 14.20 à 14.25 | courbes d'un boosting, learning rate et nombre d'arbres, frontières et débogage d'AdaBoost, AdaBoostClassifier et GradientBoostingRegressor dans mylearn.ensemble (*prochaine session de génération*) | | | |
| D | 14.26 à 14.28 | HistGradientBoosting, XGBoost et LightGBM sur California, un benchmark reproductible, le défi RMSE ≤ 0,45 (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="14")  # your own library

## Objectifs et rappel express

- Calculer à la main un vote pondéré, la probabilité qu'une majorité se trompe, la part des exemples oubliés par un bootstrap, une itération d'AdaBoost et deux itérations de gradient boosting.
- Écrire et vérifier contre scikit-learn les ensembles de ta librairie, construits sur tes arbres du ch. 13 : `plurality_vote`, `bootstrap_indices`, `BaggingClassifier`, `RandomForestClassifier`, `ExtraTreesClassifier`, `AdaBoostClassifier` et `GradientBoostingRegressor`.
- Utiliser les forêts de scikit-learn, HistGradientBoosting, XGBoost et LightGBM avec early stopping, et interpréter l'importance des features.
- Choisir entre bagging et boosting, et rendre un benchmark reproductible.

**Rappel express.** Vote pondéré : $\hat y = \arg\max_k \sum_m w_m\,\mathbb{1}[h_m(\mathbf{x}) = k]$ ; majorité de $M$ votants indépendants : $\sum_{j > M/2} \binom{M}{j} p^j (1 - p)^{M - j}$ ; jamais tiré par un bootstrap : $(1 - 1/n)^n \to e^{-1} \approx 0{,}368$ ; variance d'une moyenne corrélée : $\rho\sigma^2 + (1 - \rho)\sigma^2/M$ ; AdaBoost (SAMME) : $\varepsilon_m = \sum_i w_i \mathbb{1}[h_m(\mathbf{x}_i) \ne y_i] / \sum_i w_i$, $\alpha_m = \eta\big(\ln\frac{1 - \varepsilon_m}{\varepsilon_m} + \ln(K - 1)\big)$, $w_i \leftarrow w_i e^{\alpha_m}$ pour les mal classés ; gradient boosting : $F_0 = \bar y$, $r_i = y_i - F_{m-1}(\mathbf{x}_i)$, $F_m = F_{m-1} + \eta\, h_m$. En scikit-learn : `sklearn.ensemble` (`BaggingClassifier`, `RandomForestClassifier`, `ExtraTreesClassifier`, `AdaBoostClassifier`, `GradientBoostingRegressor`, `HistGradientBoostingRegressor`, `VotingClassifier`, `StackingClassifier`), `sklearn.inspection.permutation_importance` ; `xgboost.XGBRegressor`, `lightgbm.LGBMRegressor`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 14.1, 14.2, 14.5 à 14.7 et ∂ 14.3)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 14.4, la réflexion (🗣️ 14.8, ⚖️ 14.9, 📄 14.10) et l'entretien se corrigent avec `05_solutions.md`.

**14.Q1 — Pourquoi un groupe peut battre chacun de ses membres**

In [ ]:
# 14.Q1: the values you found on paper
answer_14_Q1a = ...  # a) True or False
answer_14_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q1c = ...  # c) True or False
answer_14_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_Q1a, answer_14_Q1b, answer_14_Q1c, answer_14_Q1d]):
    wb.check(f"14.Q1{letter}", answer)

**14.Q2 — Un ensemble corrige-t-il un biais des données ?**

In [ ]:
# 14.Q2: the values you found on paper
answer_14_Q2a = ...  # a) True or False
answer_14_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q2c = ...  # c) True or False
answer_14_Q2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_Q2a, answer_14_Q2b, answer_14_Q2c, answer_14_Q2d]):
    wb.check(f"14.Q2{letter}", answer)

**14.Q3 — Pluralité, majorité et vote pondéré**

In [ ]:
# 14.Q3: the values you found on paper
answer_14_Q3a = ...  # a) a class: "A", "B" or "C"
answer_14_Q3b = ...  # b) True or False
answer_14_Q3c = ...  # c) a number
answer_14_Q3d = ...  # d) a class: "A", "B" or "C"

for letter, answer in zip("abcd", [answer_14_Q3a, answer_14_Q3b, answer_14_Q3c, answer_14_Q3d]):
    wb.check(f"14.Q3{letter}", answer)

**14.Q4 — Tirer avec remise**

In [ ]:
# 14.Q4: the values you found on paper
answer_14_Q4a = ...  # a) True or False
answer_14_Q4b = ...  # b) a whole number
answer_14_Q4c = ...  # c) a whole number
answer_14_Q4d = ...  # d) True or False
answer_14_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q4a, answer_14_Q4b, answer_14_Q4c, answer_14_Q4d, answer_14_Q4e]):
    wb.check(f"14.Q4{letter}", answer)

**14.Q5 — Ce que le bagging réduit**

In [ ]:
# 14.Q5: the values you found on paper
answer_14_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q5c = ...  # c) True or False
answer_14_Q5d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_Q5a, answer_14_Q5b, answer_14_Q5c, answer_14_Q5d]):
    wb.check(f"14.Q5{letter}", answer)

**14.Q6 — Le hasard d'une forêt aléatoire**

In [ ]:
# 14.Q6: the values you found on paper
answer_14_Q6a = ...  # a) a whole number
answer_14_Q6b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q6c = ...  # c) True or False
answer_14_Q6d = ...  # d) two letters, e.g. "EF"
answer_14_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q6a, answer_14_Q6b, answer_14_Q6c, answer_14_Q6d, answer_14_Q6e]):
    wb.check(f"14.Q6{letter}", answer)

**14.Q7 — ExtraTrees : encore plus de hasard**

In [ ]:
# 14.Q7: the values you found on paper
answer_14_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q7b = ...  # b) True or False
answer_14_Q7c = ...  # c) True or False
answer_14_Q7d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_14_Q7a, answer_14_Q7b, answer_14_Q7c, answer_14_Q7d]):
    wb.check(f"14.Q7{letter}", answer)

**14.Q8 — Apprenant faible, souche et apprenant fort**

In [ ]:
# 14.Q8: the values you found on paper
answer_14_Q8a = ...  # a) a whole number
answer_14_Q8b = ...  # b) a whole number
answer_14_Q8c = ...  # c) True or False
answer_14_Q8d = ...  # d) the letter of your choice, e.g. "E"
answer_14_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q8a, answer_14_Q8b, answer_14_Q8c, answer_14_Q8d, answer_14_Q8e]):
    wb.check(f"14.Q8{letter}", answer)

**14.Q9 — Ce que repondère AdaBoost**

In [ ]:
# 14.Q9: the values you found on paper
answer_14_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q9b = ...  # b) a number
answer_14_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_14_Q9d = ...  # d) True or False
answer_14_Q9e = ...  # e) a number

for letter, answer in zip("abcde", [answer_14_Q9a, answer_14_Q9b, answer_14_Q9c, answer_14_Q9d, answer_14_Q9e]):
    wb.check(f"14.Q9{letter}", answer)

**14.Q10 — Bagging ou boosting ?**

In [ ]:
# 14.Q10: the values you found on paper
answer_14_Q10a = ...  # a) six letters B or O, e.g. "BBBBBB"
answer_14_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q10c = ...  # c) True or False

for letter, answer in zip("abc", [answer_14_Q10a, answer_14_Q10b, answer_14_Q10c]):
    wb.check(f"14.Q10{letter}", answer)

**14.Q11 — Gradient boosting : corriger les résidus**

In [ ]:
# 14.Q11: the values you found on paper
answer_14_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q11b = ...  # b) a number
answer_14_Q11c = ...  # c) a number
answer_14_Q11d = ...  # d) True or False
answer_14_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q11a, answer_14_Q11b, answer_14_Q11c, answer_14_Q11d, answer_14_Q11e]):
    wb.check(f"14.Q11{letter}", answer)

**14.R1 — L'impureté de Gini d'un nœud**

In [ ]:
# 14.R1: the values you found on paper
answer_14_R1a = ...  # a) a number
answer_14_R1b = ...  # b) a number with 3 decimals
answer_14_R1c = ...  # c) a number
answer_14_R1d = ...  # d) a number with 3 decimals
answer_14_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_R1a, answer_14_R1b, answer_14_R1c, answer_14_R1d, answer_14_R1e]):
    wb.check(f"14.R1{letter}", answer)

**14.R2 — Induction et généralisation**

In [ ]:
# 14.R2: the values you found on paper
answer_14_R2a = ...  # a) the letter of your choice, e.g. "E"
answer_14_R2b = ...  # b) True or False
answer_14_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_14_R2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_R2a, answer_14_R2b, answer_14_R2c, answer_14_R2d]):
    wb.check(f"14.R2{letter}", answer)

**14.R3 — Un-contre-tous et un-contre-un**

In [ ]:
# 14.R3: the values you found on paper
answer_14_R3a = ...  # a) a whole number
answer_14_R3b = ...  # b) a whole number
answer_14_R3c = ...  # c) a whole number
answer_14_R3d = ...  # d) True or False
answer_14_R3e = ...  # e) a number

for letter, answer in zip("abcde", [answer_14_R3a, answer_14_R3b, answer_14_R3c, answer_14_R3d, answer_14_R3e]):
    wb.check(f"14.R3{letter}", answer)

**Ex 14.1 — Dépouiller un vote pondéré**

In [ ]:
# Ex 14.1: the values you found on paper
answer_14_1a = ...  # a) a number
answer_14_1b = ...  # b) a class: 0, 1 or 2
answer_14_1c = ...  # c) a class: 0, 1 or 2
answer_14_1d = ...  # d) a list of four classes
answer_14_1e = ...  # e) a list of four classes
answer_14_1f = ...  # f) a whole number
answer_14_1g = ...  # g) True or False
answer_14_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_14_1a, answer_14_1b, answer_14_1c, answer_14_1d, answer_14_1e, answer_14_1f, answer_14_1g, answer_14_1h]):
    wb.check(f"14.1{letter}", answer)

**Ex 14.2 — Additionner les voix de trois droites, région par région**

In [ ]:
# Ex 14.2: the values you found on paper
answer_14_2a = ...  # a) a number
answer_14_2b = ...  # b) a number
answer_14_2c = ...  # c) a class: 0 or 1
answer_14_2d = ...  # d) a whole number
answer_14_2e = ...  # e) a list of five classes
answer_14_2f = ...  # f) a whole number
answer_14_2g = ...  # g) a number
answer_14_2h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_14_2a, answer_14_2b, answer_14_2c, answer_14_2d, answer_14_2e, answer_14_2f, answer_14_2g, answer_14_2h]):
    wb.check(f"14.2{letter}", answer)

**Ex 14.3 — Probabilité d'être oublié par un bootstrap : vers 36,8 %**

In [ ]:
# Ex 14.3: the values you found on paper
answer_14_3a = ...  # a) a number with 3 decimals
answer_14_3b = ...  # b) a number with 2 decimals
answer_14_3c = ...  # c) a number with 4 decimals
answer_14_3d = ...  # d) a number with 4 decimals
answer_14_3e = ...  # e) a number with 4 decimals
answer_14_3f = ...  # f) a number with 4 decimals
answer_14_3g = ...  # g) a number with 1 decimal
answer_14_3h = ...  # h) a number with 3 decimals

for letter, answer in zip("abcdefgh", [answer_14_3a, answer_14_3b, answer_14_3c, answer_14_3d, answer_14_3e, answer_14_3f, answer_14_3g, answer_14_3h]):
    wb.check(f"14.3{letter}", answer)

**Ex 14.5 — Quand la majorité de votants indépendants se trompe**

In [ ]:
# Ex 14.5: the values you found on paper
answer_14_5a = ...  # a) a number with 3 decimals
answer_14_5b = ...  # b) a number with 3 decimals
answer_14_5c = ...  # c) a number with 3 decimals
answer_14_5d = ...  # d) a number with 4 decimals
answer_14_5e = ...  # e) a number with 4 decimals
answer_14_5f = ...  # f) a number
answer_14_5g = ...  # g) an odd whole number
answer_14_5h = ...  # h) a number with 3 decimals
answer_14_5i = ...  # i) a number with 3 decimals

for letter, answer in zip("abcdefghi", [answer_14_5a, answer_14_5b, answer_14_5c, answer_14_5d, answer_14_5e, answer_14_5f, answer_14_5g, answer_14_5h, answer_14_5i]):
    wb.check(f"14.5{letter}", answer)

**Ex 14.6 — Une itération d'AdaBoost à la main**

In [ ]:
# Ex 14.6: the values you found on paper
answer_14_6a = ...  # a) a number
answer_14_6b = ...  # b) a number with 3 decimals
answer_14_6c = ...  # c) a number
answer_14_6d = ...  # d) a number with 4 decimals
answer_14_6e = ...  # e) a list [threshold, class on the left]
answer_14_6f = ...  # f) a number with 4 decimals
answer_14_6g = ...  # g) a number with 3 decimals
answer_14_6h = ...  # h) a list of eight classes
answer_14_6i = ...  # i) a number with 4 decimals
answer_14_6j = ...  # j) a number with 3 decimals
answer_14_6k = ...  # k) a list of eight classes
answer_14_6l = ...  # l) a number with 3 decimals
answer_14_6m = ...  # m) True or False

for letter, answer in zip("abcdefghijklm", [answer_14_6a, answer_14_6b, answer_14_6c, answer_14_6d, answer_14_6e, answer_14_6f, answer_14_6g, answer_14_6h, answer_14_6i, answer_14_6j, answer_14_6k, answer_14_6l, answer_14_6m]):
    wb.check(f"14.6{letter}", answer)

**Ex 14.7 — Deux itérations de gradient boosting sur cinq points**

In [ ]:
# Ex 14.7: the values you found on paper
answer_14_7a = ...  # a) a number
answer_14_7b = ...  # b) a list of five numbers
answer_14_7c = ...  # c) a threshold
answer_14_7d = ...  # d) a list [left value, right value], 3 decimals
answer_14_7e = ...  # e) a number with 2 decimals
answer_14_7f = ...  # f) a number with 3 decimals
answer_14_7g = ...  # g) a threshold
answer_14_7h = ...  # h) a number with 4 decimals
answer_14_7i = ...  # i) a number with 4 decimals
answer_14_7j = ...  # j) a number with 4 decimals
answer_14_7k = ...  # k) a number

for letter, answer in zip("abcdefghijk", [answer_14_7a, answer_14_7b, answer_14_7c, answer_14_7d, answer_14_7e, answer_14_7f, answer_14_7g, answer_14_7h, answer_14_7i, answer_14_7j, answer_14_7k]):
    wb.check(f"14.7{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire quand un ensemble bat ses membres, et pourquoi il ne corrige pas un biais commun ?
2. Sais-tu dérouler à la main un vote pondéré, un tour d'AdaBoost et une étape de gradient boosting ?
3. Sais-tu choisir entre une forêt et un gradient boosting, et régler le nombre d'arbres et le learning rate ?

**Pour aller plus loin** : les articles de Breiman (1996, 2001), de Freund et Schapire (1997) et de Friedman (2001), et le guide utilisateur « Ensembles » de scikit-learn, cités dans la fiche. La suite : le ch. 15 (scikit-learn), qui mettra ces ensembles dans des pipelines et des recherches d'hyperparamètres.

*Les exercices de code 14.11 à 14.28 (parties A à D, ta librairie `ensemble`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 14` pour obtenir le notebook complet.*